In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
!pip install flask -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


D:\86151\Anaconda3\Scripts\pip-script.py:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import load_entry_point


In [6]:
%pip install --upgrade pip setuptools wheel -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/8a/6a/19e9fe04fca059ccf770861c7d5721ab4c2aebc539889e97c7977528a53b/pip-24.0-py3-none-any.whl
Requirement already up-to-date: setuptools in d:\86151\anaconda3\lib\site-packages (68.0.0)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/c7/c3/55076fc728723ef927521abaa1955213d094933dc36d4a2008d5101e1af5/wheel-0.42.0-py3-none-any.whl
  Found existing installation: pip 19.0.3
    Uninstalling pip-19.0.3:
      Successfully uninstalled pip-19.0.3
  Found existing installation: wheel 0.33.1
    Uninstalling wheel-0.33.1:
      Successfully uninstalled wheel-0.33.1
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install opencv-python-headless --only-binary :all: -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     --------------------------------------- 40.1/40.1 MB 13.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


DEPRECATION: pandas 0.24.2 has a non-standard dependency specifier pytz>=2011k. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pandas or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [9]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from flask import Flask, request, jsonify
import tensorflow as tf
import numpy as np
import cv2
import os
from datetime import datetime

# 初始化Flask应用
app = Flask(__name__)

# -------------------------- 固定配置，和你的环境完全匹配 --------------------------
MODEL_PATH = r"D:\86151\bishedaima\disease_model_optimized.h5"
UPLOAD_FOLDER = r"D:\86151\bishedaima\esp32_web_upload"
CONFIDENCE_THRESHOLD = 95
IMAGE_SIZE = (128, 128)
CLASS_NAMES_CN = [
    '番茄细菌性斑点病', '番茄早疫病', '番茄健康叶片',
    '番茄晚疫病', '番茄叶霉病', '番茄壳针孢叶斑病',
    '番茄二斑叶螨虫害', '番茄靶斑病', '番茄花叶病毒病',
    '番茄黄化曲叶病毒病'
]

# 创建文件夹
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# 全局加载模型（只加载一次，提升速度）
print("🔄 正在加载AI模型...")
model = tf.keras.models.load_model(MODEL_PATH)
print("✅ AI模型加载完成！")

# -------------------------- 核心接口：ESP32/网页上传图片识别 --------------------------
@app.route('/ai_recognize', methods=['POST'])
def ai_recognize():
    try:
        # 接收ESP32上传的图片
        image_file = request.files['image']
        if not image_file:
            return jsonify({'status': 'error', 'msg': '没有收到图片'}), 400
        
        # 保存图片
        filename = datetime.now().strftime('%Y%m%d_%H%M%S') + '.jpg'
        save_path = os.path.join(UPLOAD_FOLDER, filename)
        image_file.save(save_path)

        # 读取+预处理图片
        img_bgr = cv2.imread(save_path)
        if img_bgr is None:
            return jsonify({'status': 'error', 'msg': '图片读取失败'}), 400
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, IMAGE_SIZE)
        img_normalized = img_resized / 255.0
        img_input = np.expand_dims(img_normalized, axis=0)

        # AI预测
        pred_prob = model.predict(img_input, verbose=0)[0]
        pred_class_idx = np.argmax(pred_prob)
        pred_class_name = CLASS_NAMES_CN[pred_class_idx]
        pred_confidence = round(float(pred_prob[pred_class_idx] * 100), 2)

        # 阈值判断
        if pred_confidence < CONFIDENCE_THRESHOLD:
            return jsonify({
                'status': 'unknown',
                'class_name': '非番茄叶片，无法识别',
                'confidence': pred_confidence,
                'warning': '请拍摄番茄叶片图片'
            })
        
        # 返回识别结果
        return jsonify({
            'status': 'success',
            'class_name': pred_class_name,
            'confidence': pred_confidence,
            'warning': '叶片健康，无需处理' if pred_class_name == '番茄健康叶片' else f'检测到{pred_class_name}，建议及时处理'
        })
    
    except Exception as e:
        print(f"识别出错：{e}")
        return jsonify({'status': 'error', 'msg': '识别失败'}), 500

# -------------------------- 浏览器访问的测试网页 --------------------------
@app.route('/')
def index():
    return '''
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>番茄病害AI识别服务</title>
</head>
<body style="max-width: 800px; margin: 0 auto; padding: 20px; font-family: SimHei;">
    <h1>番茄叶片病虫害AI识别服务</h1>
    <input type="file" id="image-input" accept="image/*">
    <button onclick="recognize()" style="padding: 10px 20px; margin: 20px 0;">开始识别</button>
    <div id="result"></div>
    <script>
        async function recognize() {
            const input = document.getElementById('image-input');
            if (!input.files[0]) {
                alert("请先选择图片");
                return;
            }
            const formData = new FormData();
            formData.append('image', input.files[0]);
            const res = await fetch('/ai_recognize', {method: 'POST', body: formData});
            const result = await res.json();
            document.getElementById('result').innerHTML = `<h3>识别结果：${result.class_name}，置信度：${result.confidence}%</h3><p>${result.warning}</p>`;
        }
    </script>
</body>
</html>
    '''

# 启动服务，0.0.0.0允许局域网所有设备（包括ESP32）访问
# 启动服务：先本地测试，避开编码问题
if __name__ == '__main__':
    # 先加这2行，解决Windows编码问题
    import os
    os.environ['PYTHONUTF8'] = '1'
    # 先用127.0.0.1本地测试，端口改成8090（避开占用）
    app.run(host='127.0.0.1', port=8090, debug=False)

🔄 正在加载AI模型...
✅ AI模型加载完成！
 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xba in position 2: invalid start byte

In [10]:
%pip install waitress -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 57.7/57.7 kB 1.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


DEPRECATION: pandas 0.24.2 has a non-standard dependency specifier pytz>=2011k. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pandas or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [6]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from flask import Flask, request, jsonify
import tensorflow as tf
import numpy as np
import cv2
import os
from datetime import datetime
# 导入waitress，替代Flask自带服务器
from waitress import serve

# 初始化Flask应用
app = Flask(__name__)

# -------------------------- 固定配置，和你的环境完全匹配 --------------------------
# 模型路径固定不变
MODEL_PATH = r"D:\86151\bishedaima\disease_model_optimized.h5"
# 上传图片保存文件夹
UPLOAD_FOLDER = r"D:\bishe\esp32_photos_auto"
# 置信度阈值，和你单张代码同步，改成50（可以自己调整，比如60、70）
CONFIDENCE_THRESHOLD = 50
# 图片输入尺寸，和训练一致
IMAGE_SIZE = (128, 128)
# 病害类别名，和训练完全一致
CLASS_NAMES_CN = [
    '番茄细菌性斑点病', '番茄早疫病', '番茄健康叶片',
    '番茄晚疫病', '番茄叶霉病', '番茄壳针孢叶斑病',
    '番茄二斑叶螨虫害', '番茄靶斑病', '番茄花叶病毒病',
    '番茄黄化曲叶病毒病'
]

# 创建文件夹（不存在就自动创建，不用手动建）
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# -------------------------- 核心接口：网页/ESP32上传图片识别 --------------------------
@app.route('/ai_recognize', methods=['POST'])
def ai_recognize():
    try:
        # 1. 接收前端/ESP32上传的图片
        image_file = request.files['image']
        if not image_file:
            return jsonify({'status': 'error', 'msg': '没有收到图片'}), 400
        
        # 2. 保存图片到本地
        filename = datetime.now().strftime('%Y%m%d_%H%M%S') + '.jpg'
        save_path = os.path.join(UPLOAD_FOLDER, filename)
        image_file.save(save_path)
        print(f"✅ 图片已保存：{save_path}")

        # 3. 读取+预处理图片
        img_bgr = cv2.imread(save_path)
        if img_bgr is None:
            return jsonify({'status': 'error', 'msg': '图片读取失败，文件损坏'}), 400
        # 转RGB格式
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # 缩放到训练尺寸
        img_resized = cv2.resize(img_rgb, IMAGE_SIZE)
        # 像素归一化
        img_normalized = img_resized / 255.0
        # 增加batch维度，符合模型输入要求
        img_input = np.expand_dims(img_normalized, axis=0)

        # 4. 【关键修复】每次请求都加载模型，彻底避开多线程计算图问题
        print("🔄 正在加载AI模型...")
        model = tf.keras.models.load_model(MODEL_PATH)
        print("✅ AI模型加载完成，开始预测...")
        
        # 5. AI预测
        pred_prob = model.predict(img_input, verbose=0)[0]
        pred_class_idx = np.argmax(pred_prob)
        pred_class_name = CLASS_NAMES_CN[pred_class_idx]
        pred_confidence = round(float(pred_prob[pred_class_idx] * 100), 2)
        print(f"📊 预测结果：{pred_class_name}，置信度：{pred_confidence}%")

        # 6. 阈值判断，返回结果
        if pred_confidence < CONFIDENCE_THRESHOLD:
            return jsonify({
                'status': 'unknown',
                'class_name': '非番茄叶片，无法识别',
                'confidence': pred_confidence,
                'warning': '请拍摄清晰的番茄叶片图片'
            })
        
        # 识别成功，返回结果
        return jsonify({
            'status': 'success',
            'class_name': pred_class_name,
            'confidence': pred_confidence,
            'warning': '叶片健康，无需处理' if pred_class_name == '番茄健康叶片' else f'检测到【{pred_class_name}】，建议及时摘除病叶并喷施对应药剂'
        })
    
    except Exception as e:
        # 打印详细错误到控制台，方便排查
        print(f"❌ 识别出错：{str(e)}")
        return jsonify({'status': 'error', 'msg': f'识别失败：{str(e)}'}), 500

# -------------------------- 浏览器访问的前端网页 --------------------------
@app.route('/')
def index():
    return '''
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>番茄叶片病虫害AI识别服务</title>
    <style>
        * {margin: 0;padding: 0;box-sizing: border-box;font-family: "SimHei", sans-serif;}
        body {max-width: 800px; margin: 0 auto; padding: 40px 20px;}
        h1 {text-align: center; margin-bottom: 40px; color: #2c3e50;}
        .upload-area {display: flex; align-items: center; gap: 20px; margin-bottom: 30px; flex-wrap: wrap;}
        button {padding: 10px 25px; border: none; border-radius: 5px; cursor: pointer; font-size: 16px;}
        .btn-primary {background-color: #3498db; color: white;}
        .btn-primary:hover {background-color: #2980b9;}
        .result-box {margin-top: 30px; padding: 20px; border-radius: 8px; background: #f8f9fa;}
        .result-title {font-size: 20px; font-weight: bold; margin-bottom: 10px;}
        .result-success {color: #27ae60;}
        .result-error {color: #e74c3c;}
        .result-unknown {color: #7f8c8d;}
    </style>
</head>
<body>
    <h1>番茄叶片病虫害AI识别服务</h1>
    
    <div class="upload-area">
        <input type="file" id="image-input" accept="image/*">
        <button class="btn-primary" onclick="recognize()">开始识别</button>
    </div>

    <div class="result-box" id="result-box">
        <div class="result-title" id="result-title">请选择图片并点击识别</div>
        <p id="result-desc"></p>
    </div>

    <script>
        async function recognize() {
            const input = document.getElementById('image-input');
            const titleBox = document.getElementById('result-title');
            const descBox = document.getElementById('result-desc');
            
            if (!input.files[0]) {
                titleBox.textContent = "请先选择图片";
                titleBox.className = "result-title result-error";
                descBox.textContent = "";
                return;
            }

            titleBox.textContent = "正在识别中...";
            titleBox.className = "result-title";
            descBox.textContent = "";

            const formData = new FormData();
            formData.append('image', input.files[0]);
            
            try {
                const res = await fetch('/ai_recognize', {method: 'POST', body: formData});
                const result = await res.json();

                if (result.status === 'success') {
                    titleBox.textContent = `识别结果：${result.class_name} | 置信度：${result.confidence}%`;
                    titleBox.className = "result-title result-success";
                    descBox.textContent = result.warning;
                } else if (result.status === 'unknown') {
                    titleBox.textContent = `无法识别 | 最高置信度：${result.confidence}%`;
                    titleBox.className = "result-title result-unknown";
                    descBox.textContent = result.warning;
                } else {
                    titleBox.textContent = "识别失败";
                    titleBox.className = "result-title result-error";
                    descBox.textContent = result.msg;
                }
            } catch (e) {
                titleBox.textContent = "识别出错";
                titleBox.className = "result-title result-error";
                descBox.textContent = "网络错误，请检查服务是否正常运行";
                console.error(e);
            }
        }
    </script>
</body>
</html>
    '''

# 启动服务：本地测试用，100%兼容Windows
if __name__ == '__main__':
    print("🚀 AI识别服务已启动！")
    print("📱 本地访问地址：http://127.0.0.1:8099")
    # 只用本地回环地址，端口8099，避开占用和编码问题
    serve(app, host='127.0.0.1', port=8099)

🚀 AI识别服务已启动！
📱 本地访问地址：http://127.0.0.1:8099
✅ 图片已保存：D:\bishe\esp32_photos_auto\20260315_233327.jpg
🔄 正在加载AI模型...
✅ AI模型加载完成，开始预测...
📊 预测结果：番茄黄化曲叶病毒病，置信度：98.57%
✅ 图片已保存：D:\bishe\esp32_photos_auto\20260315_233503.jpg
🔄 正在加载AI模型...
✅ AI模型加载完成，开始预测...
📊 预测结果：番茄细菌性斑点病，置信度：52.31%
